# Grouping Feature Ablation: Primary-Only 排序

这个 notebook 读取 `experiment_metric_summary/grouping_feature_ablation_primary_only_sorted.csv`，单独查看 5-feature 分组和之前大特征/全特征分组的 Primary-Only 对比结果。

排序依据：`primary_rank` 越小越好。

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

repo_dir = Path.cwd()
if repo_dir.name == "notebooks":
    repo_dir = repo_dir.parent

csv_path = repo_dir / "experiment_metric_summary" / "grouping_feature_ablation_primary_only_sorted.csv"
df = pd.read_csv(csv_path)

df["primary_rank"] = df["primary_rank"].astype(int)
df["overall_rank"] = df["overall_rank"].astype(int)

display(Markdown(f"Loaded `{csv_path}` with **{len(df)}** rows."))

## 总表

包含 5f compact 和之前较大/全特征分组结果。

In [ ]:
show_cols = [
    "primary_rank",
    "primary_rank_score",
    "overall_rank",
    "architecture",
    "grouping_method_short",
    "grouping_feature_set_short",
    "feature_group",
    "primary_load_cv_rmse_pct",
    "primary_abs_nmbe_pct",
    "primary_comfort_exceedance_pct",
    "test_reward_sum",
    "experiment",
]

display(
    df[show_cols]
    .sort_values(["primary_rank", "primary_rank_score"])
    .style
    .background_gradient(subset=["primary_rank", "primary_rank_score", "primary_load_cv_rmse_pct", "primary_abs_nmbe_pct", "primary_comfort_exceedance_pct"], cmap="RdYlGn_r")
    .format({
        "primary_rank_score": "{:.4f}",
        "primary_load_cv_rmse_pct": "{:.4f}",
        "primary_abs_nmbe_pct": "{:.4f}",
        "primary_comfort_exceedance_pct": "{:.4f}",
        "test_reward_sum": "{:.2f}",
    })
)

## 只看 5-feature compact

5f 特征：`bes_capacity_kwh`, `hvac_total_kw`, `heating_mean`, `nsl_mean`, `comfort_lower_excess_mean`。

In [ ]:
df_5f = df[df["feature_group"] == "5f compact"].copy()
display(
    df_5f[show_cols]
    .sort_values(["primary_rank", "primary_rank_score"])
    .style
    .background_gradient(subset=["primary_rank", "primary_rank_score", "primary_load_cv_rmse_pct", "primary_abs_nmbe_pct", "primary_comfort_exceedance_pct"], cmap="RdYlGn_r")
    .format({
        "primary_rank_score": "{:.4f}",
        "primary_load_cv_rmse_pct": "{:.4f}",
        "primary_abs_nmbe_pct": "{:.4f}",
        "primary_comfort_exceedance_pct": "{:.4f}",
        "test_reward_sum": "{:.2f}",
    })
)

## 之前的大特征/全特征分组

包含：`static_extended`, `operational_profile`, `static_operational`。

In [ ]:
df_full = df[df["feature_group"] == "previous larger/full"].copy()
display(
    df_full[show_cols]
    .sort_values(["primary_rank", "primary_rank_score"])
    .style
    .background_gradient(subset=["primary_rank", "primary_rank_score", "primary_load_cv_rmse_pct", "primary_abs_nmbe_pct", "primary_comfort_exceedance_pct"], cmap="RdYlGn_r")
    .format({
        "primary_rank_score": "{:.4f}",
        "primary_load_cv_rmse_pct": "{:.4f}",
        "primary_abs_nmbe_pct": "{:.4f}",
        "primary_comfort_exceedance_pct": "{:.4f}",
        "test_reward_sum": "{:.2f}",
    })
)

## 按架构和分组方法汇总

这里看每种组合的最好 `primary_rank` 和平均指标。

In [ ]:
summary = (
    df.groupby(["architecture", "grouping_method_short", "grouping_feature_set_short", "feature_group"], as_index=False)
    .agg(
        best_primary_rank=("primary_rank", "min"),
        mean_primary_rank_score=("primary_rank_score", "mean"),
        mean_cv_rmse=("primary_load_cv_rmse_pct", "mean"),
        mean_abs_nmbe=("primary_abs_nmbe_pct", "mean"),
        mean_comfort=("primary_comfort_exceedance_pct", "mean"),
        mean_reward=("test_reward_sum", "mean"),
    )
    .sort_values(["best_primary_rank", "mean_primary_rank_score"])
)

display(
    summary.style
    .background_gradient(subset=["best_primary_rank", "mean_primary_rank_score", "mean_cv_rmse", "mean_abs_nmbe", "mean_comfort"], cmap="RdYlGn_r")
    .format({
        "mean_primary_rank_score": "{:.4f}",
        "mean_cv_rmse": "{:.4f}",
        "mean_abs_nmbe": "{:.4f}",
        "mean_comfort": "{:.4f}",
        "mean_reward": "{:.2f}",
    })
)

## 图：Primary rank score

`primary_rank_score` 越低越好。

In [ ]:
plot_df = df.sort_values("primary_rank_score", ascending=True).copy()
plot_df["label"] = (
    plot_df["architecture"].str.replace(" global", "", regex=False)
    + " | " + plot_df["grouping_method_short"]
    + " | " + plot_df["grouping_feature_set_short"]
)
colors = plot_df["feature_group"].map({"5f compact": "#3b82f6", "previous larger/full": "#9ca3af"})

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(plot_df["label"], plot_df["primary_rank_score"], color=colors)
ax.invert_yaxis()
ax.set_xlabel("Primary rank score lower is better")
ax.set_title("Grouping feature ablation: Primary rank score")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## 图：CV-RMSE vs Comfort

横轴是 load tracking CV-RMSE，纵轴是 comfort exceedance，两个都越低越好。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for feature_group, group_df in df.groupby("feature_group"):
    marker = "o" if feature_group == "5f compact" else "s"
    ax.scatter(
        group_df["primary_load_cv_rmse_pct"],
        group_df["primary_comfort_exceedance_pct"],
        label=feature_group,
        marker=marker,
        s=80,
        alpha=0.85,
    )

for _, row in df.iterrows():
    label = f"{row['architecture'].split()[0]}-{row['grouping_method_short']}-{row['grouping_feature_set_short']}"
    ax.annotate(label, (row["primary_load_cv_rmse_pct"], row["primary_comfort_exceedance_pct"]), fontsize=8, alpha=0.75)

ax.set_xlabel("Primary load CV-RMSE pct lower is better")
ax.set_ylabel("Primary comfort exceedance pct lower is better")
ax.set_title("Load tracking vs comfort")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## 快速结论辅助

下面自动列出最好的 5 个结果。

In [ ]:
top = df.sort_values(["primary_rank", "primary_rank_score"]).head(5)
display(top[[
    "primary_rank",
    "architecture",
    "grouping_method_short",
    "grouping_feature_set_short",
    "primary_load_cv_rmse_pct",
    "primary_abs_nmbe_pct",
    "primary_comfort_exceedance_pct",
    "test_reward_sum",
    "experiment",
]])